# Analisis Klasifikasi Kesuburan Tanah dengan K-Nearest Neighbors (KNN)

Analisis ini mengklasifikasikan kesuburan tanah menjadi **Subur** dan **Tidak Subur** 
menggunakan algoritma **K-Nearest Neighbors (KNN)** berdasarkan 10 fitur agronomis tanah.

## 1. Import Library

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import (accuracy_score, precision_score,
                              recall_score, f1_score,
                              classification_report, confusion_matrix)
import matplotlib.pyplot as plt
import seaborn as sns

print('Library berhasil diimport!')

Library berhasil diimport!


## 2. Load Dataset

In [ ]:
df = pd.read_csv('dataset_kesuburan_tanah_missing.csv')
print(f'Shape dataset: {df.shape}')
print(f'\nDistribusi Kelas:')
print(df['Label'].value_counts())
df.head()

Shape dataset: (2000, 12)

Distribusi Kelas:
Label
Tidak Subur    1000
Subur          1000
Name: count, dtype: int64


## 3. Pemrosesan Data (Preprocessing)

### 3.1 Cek Missing Values

In [ ]:
df_clean = df.drop(columns=['ID'])

print('Jumlah Missing Values per Fitur:')
print('=' * 40)
mv = df_clean.isnull().sum()
print(mv[mv > 0])
print(f'\nTotal missing values: {mv.sum()}')

Jumlah Missing Values per Fitur:
N Total (%)              160
P Tersedia (ppm)         240
K Tersedia (meq/100g)    140
C Organik (%)            200
Tekstur Tanah            100
Kadar Air (%)            180
Bulk Density (g/cm³)     120
dtype: int64

Total missing values: 1140


### 3.2 Encoding Fitur Kategorikal (Tekstur Tanah)

In [ ]:
le_tekstur = LabelEncoder()
df_clean['Tekstur Tanah'] = le_tekstur.fit_transform(df_clean['Tekstur Tanah'].astype(str))

print('Mapping Tekstur Tanah:')
for i, kelas in enumerate(le_tekstur.classes_):
    print(f'  {kelas} -> {i}')

Mapping Tekstur Tanah:
  Debu -> 0
  Lempung -> 1
  Lempung Berpasir -> 2
  Lempung Berliat -> 3
  Liat -> 4
  Pasir -> 5
  nan -> 6


### 3.3 Imputasi Missing Values (Median)

In [ ]:
X = df_clean.drop(columns=['Label'])
y = df_clean['Label']

imputer = SimpleImputer(strategy='median')
X_imputed = imputer.fit_transform(X)

print(f'Missing values sebelum imputasi: {X.isnull().sum().sum()}')
print(f'Missing values setelah imputasi: {pd.DataFrame(X_imputed).isnull().sum().sum()}')

Missing values sebelum imputasi: 1140
Missing values setelah imputasi: 0


### 3.4 Normalisasi Fitur (StandardScaler)

In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_imputed)

print('Normalisasi selesai.')
print(f'Mean fitur pertama setelah scaling: {X_scaled[:,0].mean():.6f}')
print(f'Std  fitur pertama setelah scaling: {X_scaled[:,0].std():.6f}')

Normalisasi selesai.
Mean fitur pertama setelah scaling: 0.000000
Std  fitur pertama setelah scaling: 1.000000


### 3.5 Encoding Label & Split Data (80/20)

In [ ]:
le_y = LabelEncoder()
y_enc = le_y.fit_transform(y)
# Subur=0, Tidak Subur=1

X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y_enc,
    test_size=0.2,
    random_state=42,
    stratify=y_enc
)

print(f'Jumlah data training : {len(X_train)} sampel')
print(f'Jumlah data testing  : {len(X_test)} sampel')

Jumlah data training : 1600 sampel
Jumlah data testing  : 400 sampel


## 4. Pemodelan K-Nearest Neighbors (KNN)

In [ ]:
knn = KNeighborsClassifier(n_neighbors=5)
knn.fit(X_train, y_train)

y_pred = knn.predict(X_test)
print('Model KNN (k=5) berhasil dilatih dan melakukan prediksi.')

Model KNN (k=5) berhasil dilatih dan melakukan prediksi.


## 5. Evaluasi Model

### 5.1 Metrik Evaluasi

In [ ]:
acc  = accuracy_score(y_test, y_pred)
prec = precision_score(y_test, y_pred, average='weighted')
rec  = recall_score(y_test, y_pred, average='weighted')
f1   = f1_score(y_test, y_pred, average='weighted')

print('=' * 45)
print('        HASIL EVALUASI MODEL KNN (k=5)')
print('=' * 45)
print(f'  Accuracy  : {acc:.4f}  ({acc*100:.2f}%)')
print(f'  Precision : {prec:.4f}  ({prec*100:.2f}%)')
print(f'  Recall    : {rec:.4f}  ({rec*100:.2f}%)')
print(f'  F1-Score  : {f1:.4f}  ({f1*100:.2f}%)')
print('=' * 45)

        HASIL EVALUASI MODEL KNN (k=5)
  Accuracy  : 1.0000  (100.00%)
  Precision : 1.0000  (100.00%)
  Recall    : 1.0000  (100.00%)
  F1-Score  : 1.0000  (100.00%)


### 5.2 Classification Report

In [ ]:
print(classification_report(y_test, y_pred, target_names=le_y.classes_))

              precision    recall  f1-score   support

       Subur       1.00      1.00      1.00       200
 Tidak Subur       1.00      1.00      1.00       200

    accuracy                           1.00       400
   macro avg       1.00      1.00      1.00       400
weighted avg       1.00      1.00      1.00       400



### 5.3 Confusion Matrix

In [ ]:
cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=le_y.classes_,
            yticklabels=le_y.classes_,
            linewidths=0.5)
plt.title('Confusion Matrix - KNN (k=5)', fontsize=14, fontweight='bold')
plt.xlabel('Prediksi', fontsize=12)
plt.ylabel('Aktual', fontsize=12)
plt.tight_layout()
plt.show()
print('\nConfusion Matrix (raw):')
print(cm)


Confusion Matrix (raw):
[[200   0]
 [  0 200]]


### 5.4 Perbandingan Nilai K (Cross-Validation 5-Fold)

In [ ]:
print(f"{'K':<6} {'CV Accuracy':<15} {'Std Dev'}")
print('-' * 38)
k_values = [3, 5, 7, 9, 11]
cv_scores = []
for k in k_values:
    knn_k = KNeighborsClassifier(n_neighbors=k)
    scores = cross_val_score(knn_k, X_scaled, y_enc, cv=5, scoring='accuracy')
    cv_scores.append(scores.mean())
    print(f"k={k:<4}  {scores.mean():.4f}         ± {scores.std():.4f}")

# Plot
plt.figure(figsize=(7, 4))
plt.plot(k_values, cv_scores, marker='o', color='royalblue', linewidth=2)
plt.title('Akurasi CV vs Nilai K', fontsize=13, fontweight='bold')
plt.xlabel('Nilai K')
plt.ylabel('Akurasi')
plt.ylim([0.98, 1.01])
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

K      CV Accuracy      Std Dev
--------------------------------------
k=3    1.0000         ± 0.0000
k=5    1.0000         ± 0.0000
k=7    1.0000         ± 0.0000
k=9    1.0000         ± 0.0000
k=11   1.0000         ± 0.0000


## 6. Kesimpulan

| Metrik | Nilai | Persentase |
|--------|-------|------------|
| **Accuracy** | 1.0000 | **100.00%** |
| **Precision** | 1.0000 | **100.00%** |
| **Recall** | 1.0000 | **100.00%** |
| **F1-Score** | 1.0000 | **100.00%** |

Model **KNN (k=5)** berhasil mengklasifikasikan kesuburan tanah dengan performa sempurna. 
Fitur-fitur agronomis (pH, N, P, K, C Organik, KTK, Kejenuhan Basa, Tekstur, Kadar Air, Bulk Density) 
terbukti sangat diskriminatif dalam memisahkan kelas **Subur** dan **Tidak Subur**.

**Pipeline Preprocessing yang digunakan:**
1. Drop kolom ID
2. Label Encoding pada fitur `Tekstur Tanah`
3. Imputasi Median untuk missing values
4. StandardScaler untuk normalisasi
5. Stratified Train-Test Split 80:20